In [2]:
# =====================================
# CONSTRUCCIÓN DE LA CAPA GOLD
# Consolidación de restaurantes enriquecidos en tabla única
# =====================================

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.spatial import cKDTree

# Rutas
PROCESSED_DIR = Path("../data/processed")
GOLD_DIR = Path("../data/gold")
GOLD_DIR.mkdir(parents=True, exist_ok=True)

print("Librerías cargadas y rutas configuradas")
print(f"Processed: {PROCESSED_DIR.resolve()}")
print(f"Gold: {GOLD_DIR.resolve()}")

Librerías cargadas y rutas configuradas
Processed: /Users/juana/Desktop/tfm-data-science-gtm/data/processed
Gold: /Users/juana/Desktop/tfm-data-science-gtm/data/gold


In [3]:
# =====================================
# CARGAR FICHEROS DE PROCESSED
# =====================================

# Los tres ficheros que guardamos en la sesión anterior
enriq_direccion = pd.read_parquet(PROCESSED_DIR / "restaurantes_enriquecidos_direccion.parquet")
enriq_proximidad = pd.read_parquet(PROCESSED_DIR / "restaurantes_enriquecidos_proximidad.parquet")
censo_rest = pd.read_parquet(PROCESSED_DIR / "censo_restaurantes_madrid.parquet")

print("Ficheros cargados:")
print(f"  Enriquecidos por dirección:  {len(enriq_direccion)} filas × {len(enriq_direccion.columns)} columnas")
print(f"  Enriquecidos por proximidad: {len(enriq_proximidad)} filas × {len(enriq_proximidad.columns)} columnas")
print(f"  Censo restaurantes:          {len(censo_rest)} filas × {len(censo_rest.columns)} columnas")

Ficheros cargados:
  Enriquecidos por dirección:  818 filas × 70 columnas
  Enriquecidos por proximidad: 737 filas × 27 columnas
  Censo restaurantes:          1626 filas × 52 columnas


In [4]:
# =====================================
# INSPECCIONAR COLUMNAS DE CADA FUENTE
# =====================================

print("=== ENRIQUECIDOS POR DIRECCIÓN (columnas) ===")
print(list(enriq_direccion.columns))

print("\n=== ENRIQUECIDOS POR PROXIMIDAD (columnas) ===")
print(list(enriq_proximidad.columns))

=== ENRIQUECIDOS POR DIRECCIÓN (columnas) ===
['osm_id', 'osm_type', 'lat', 'lon', 'name', 'cuisine', 'addr_street', 'addr_housenumber', 'addr_postcode', 'addr_city', 'phone', 'website', 'opening_hours', 'outdoor_seating', 'takeaway', 'delivery', 'wheelchair', 'calle_norm_osm', 'numero_norm_osm', 'clave_cruce', 'id_local', 'id_distrito_local', 'desc_distrito_local', 'id_barrio_local', 'desc_barrio_local', 'cod_barrio_local', 'id_seccion_censal_local', 'desc_seccion_censal_local', 'coordenada_x_local', 'coordenada_y_local', 'id_tipo_acceso_local', 'desc_tipo_acceso_local', 'id_situacion_local', 'desc_situacion_local', 'id_vial_edificio', 'clase_vial_edificio', 'desc_vial_edificio', 'id_ndp_edificio', 'id_clase_ndp_edificio', 'nom_edificio', 'num_edificio', 'cal_edificio', 'secuencial_local_PC', 'id_vial_acceso', 'clase_vial_acceso', 'desc_vial_acceso', 'id_ndp_acceso', 'id_clase_ndp_acceso', 'nom_acceso', 'num_acceso', 'cal_acceso', 'coordenada_x_agrupacion', 'coordenada_y_agrupacion', 

In [5]:
# =====================================
# UNIFICAR TABLAS ENRIQUECIDAS
# =====================================

# Columnas base de OSM (están en ambas tablas)
cols_osm = ['osm_id', 'osm_type', 'lat', 'lon', 'name', 'cuisine',
            'addr_street', 'addr_housenumber', 'addr_postcode', 'addr_city',
            'phone', 'website', 'opening_hours', 'outdoor_seating',
            'takeaway', 'delivery', 'wheelchair']

# Columnas del censo (están en ambas, aunque proximidad tiene menos)
cols_censo = ['desc_barrio_local', 'desc_distrito_local', 'desc_epigrafe',
              'id_seccion_censal_local', 'metodo_match']

# La tabla de dirección tiene además el tipo de acceso; la de proximidad no
# Añadimos tipo de acceso solo si existe
cols_direccion = cols_osm + cols_censo
if 'desc_tipo_acceso_local' in enriq_direccion.columns:
    cols_direccion = cols_direccion + ['desc_tipo_acceso_local']

# Preparar cada tabla con las columnas comunes
tabla_dir = enriq_direccion[cols_direccion].copy()

tabla_prox = enriq_proximidad[cols_osm + cols_censo].copy()
tabla_prox['desc_tipo_acceso_local'] = np.nan  # no disponible en proximidad

# Unir las dos
enriquecidos = pd.concat([tabla_dir, tabla_prox], ignore_index=True)

print(f"Total restaurantes enriquecidos unificados: {len(enriquecidos)}")
print(f"  Por dirección:  {(enriquecidos['metodo_match'] == 'direccion').sum()}")
print(f"  Por proximidad: {(enriquecidos['metodo_match'] == 'proximidad').sum()}")
print(f"\nColumnas: {len(enriquecidos.columns)}")

Total restaurantes enriquecidos unificados: 1555
  Por dirección:  818
  Por proximidad: 737

Columnas: 23


In [6]:
# =====================================
# AÑADIR RESTAURANTES SIN ENRIQUECER
# =====================================

# Cargar el fichero original de OSM (los 1.688 restaurantes)
raw_osm = pd.read_parquet("../data/raw/restaurantes_centro_madrid_osm.parquet")
print(f"Total restaurantes OSM originales: {len(raw_osm)}")

# Identificar los osm_id que YA están enriquecidos
ids_enriquecidos = set(enriquecidos['osm_id'])

# Filtrar los que NO están enriquecidos
sin_enriquecer = raw_osm[~raw_osm['osm_id'].isin(ids_enriquecidos)].copy()
print(f"Restaurantes sin enriquecer: {len(sin_enriquecer)}")

# Añadir las columnas del censo como vacías (no tienen match)
sin_enriquecer['desc_barrio_local'] = np.nan
sin_enriquecer['desc_distrito_local'] = np.nan
sin_enriquecer['desc_epigrafe'] = np.nan
sin_enriquecer['id_seccion_censal_local'] = np.nan
sin_enriquecer['desc_tipo_acceso_local'] = np.nan
sin_enriquecer['metodo_match'] = 'no_matcheado'

# Quedarnos solo con las columnas que tiene la tabla enriquecidos
sin_enriquecer = sin_enriquecer[enriquecidos.columns].copy()

# Unir todo: enriquecidos + sin enriquecer = universo completo
gold_base = pd.concat([enriquecidos, sin_enriquecer], ignore_index=True)

print(f"\n=== UNIVERSO COMPLETO ===")
print(f"Total restaurantes en gold_base: {len(gold_base)}")
print(f"  Por dirección:   {(gold_base['metodo_match'] == 'direccion').sum()}")
print(f"  Por proximidad:  {(gold_base['metodo_match'] == 'proximidad').sum()}")
print(f"  Sin matchear:    {(gold_base['metodo_match'] == 'no_matcheado').sum()}")

Total restaurantes OSM originales: 1688
Restaurantes sin enriquecer: 331

=== UNIVERSO COMPLETO ===
Total restaurantes en gold_base: 1886
  Por dirección:   818
  Por proximidad:  737
  Sin matchear:    331


In [7]:
# =====================================
# DIAGNÓSTICO DE DUPLICADOS
# =====================================

# ¿Hay osm_id duplicados en la tabla de enriquecidos unificada?
dup_enriq = enriquecidos['osm_id'].duplicated().sum()
print(f"Duplicados dentro de 'enriquecidos': {dup_enriq}")

# ¿Hay solapamiento entre dirección y proximidad?
ids_dir = set(enriq_direccion['osm_id'])
ids_prox = set(enriq_proximidad['osm_id'])
solapamiento = ids_dir & ids_prox
print(f"osm_id que están en AMBAS tablas (dirección y proximidad): {len(solapamiento)}")

# ¿Cuántos osm_id únicos hay en total en enriquecidos?
print(f"osm_id únicos en enriquecidos: {enriquecidos['osm_id'].nunique()} (de {len(enriquecidos)} filas)")

# ¿Duplicados en el raw original?
print(f"osm_id únicos en raw OSM: {raw_osm['osm_id'].nunique()} (de {len(raw_osm)} filas)")

Duplicados dentro de 'enriquecidos': 198
osm_id que están en AMBAS tablas (dirección y proximidad): 0
osm_id únicos en enriquecidos: 1357 (de 1555 filas)
osm_id únicos en raw OSM: 1688 (de 1688 filas)


In [8]:
# =====================================
# ¿DÓNDE ESTÁN LOS DUPLICADOS?
# =====================================

# Duplicados en la tabla de dirección
dup_dir = enriq_direccion['osm_id'].duplicated().sum()
print(f"Duplicados en enriq_direccion: {dup_dir}")

# Duplicados en la tabla de proximidad
dup_prox = enriq_proximidad['osm_id'].duplicated().sum()
print(f"Duplicados en enriq_proximidad: {dup_prox}")

# Ver un ejemplo de restaurante duplicado en dirección
if dup_dir > 0:
    ids_duplicados = enriq_direccion[enriq_direccion['osm_id'].duplicated(keep=False)]['osm_id'].unique()
    ejemplo_id = ids_duplicados[0]
    print(f"\n=== Ejemplo de duplicado (osm_id={ejemplo_id}) ===")
    cols_mostrar = ['osm_id', 'name', 'addr_street', 'rotulo', 'desc_epigrafe', 'desc_barrio_local']
    cols_existentes = [c for c in cols_mostrar if c in enriq_direccion.columns]
    print(enriq_direccion[enriq_direccion['osm_id'] == ejemplo_id][cols_existentes].to_string())

Duplicados en enriq_direccion: 198
Duplicados en enriq_proximidad: 0

=== Ejemplo de duplicado (osm_id=26065699) ===
     osm_id           name          addr_street        rotulo    desc_epigrafe     desc_barrio_local
1  26065699  Honest Greens  Calle de Fuencarral      LA MUCCA  BAR RESTAURANTE  UNIVERSIDAD         
2  26065699  Honest Greens  Calle de Fuencarral    SIN RÓTULO  BAR RESTAURANTE  UNIVERSIDAD         
3  26065699  Honest Greens  Calle de Fuencarral  HONEST GREEN      RESTAURANTE  UNIVERSIDAD         


In [9]:
# =====================================
# DEDUPLICAR: MEJOR MATCH POR SIMILITUD DE NOMBRE
# =====================================

from difflib import SequenceMatcher

def similitud(a, b):
    """Similitud entre dos cadenas (0 a 1)."""
    if pd.isna(a) or pd.isna(b):
        return 0
    return SequenceMatcher(None, str(a).upper(), str(b).upper()).ratio()

# Para la tabla de dirección, calcular similitud entre name (OSM) y rotulo (censo)
enriq_direccion = enriq_direccion.copy()
enriq_direccion['sim_nombre'] = enriq_direccion.apply(
    lambda row: similitud(row['name'], row['rotulo']), axis=1
)

# Para cada osm_id, quedarnos con la fila de mayor similitud
enriq_direccion_dedup = (
    enriq_direccion
    .sort_values('sim_nombre', ascending=False)
    .drop_duplicates(subset='osm_id', keep='first')
    .copy()
)

print(f"Antes de deduplicar: {len(enriq_direccion)} filas")
print(f"Después de deduplicar: {len(enriq_direccion_dedup)} filas")
print(f"osm_id únicos: {enriq_direccion_dedup['osm_id'].nunique()}")

# Ver cómo quedó el ejemplo de Honest Greens
print(f"\n=== Honest Greens tras deduplicar ===")
cols_mostrar = ['osm_id', 'name', 'rotulo', 'desc_epigrafe', 'sim_nombre']
print(enriq_direccion_dedup[enriq_direccion_dedup['osm_id'] == 26065699][cols_mostrar].to_string())

Antes de deduplicar: 818 filas
Después de deduplicar: 620 filas
osm_id únicos: 620

=== Honest Greens tras deduplicar ===
     osm_id           name        rotulo desc_epigrafe  sim_nombre
3  26065699  Honest Greens  HONEST GREEN   RESTAURANTE        0.96


In [10]:
# =====================================
# REHACER UNIFICACIÓN CON TABLA DEDUPLICADA
# =====================================

# Columnas base
cols_osm = ['osm_id', 'osm_type', 'lat', 'lon', 'name', 'cuisine',
            'addr_street', 'addr_housenumber', 'addr_postcode', 'addr_city',
            'phone', 'website', 'opening_hours', 'outdoor_seating',
            'takeaway', 'delivery', 'wheelchair']
cols_censo = ['desc_barrio_local', 'desc_distrito_local', 'desc_epigrafe',
              'id_seccion_censal_local', 'metodo_match']

# Tabla dirección deduplicada
cols_dir = cols_osm + cols_censo
if 'desc_tipo_acceso_local' in enriq_direccion_dedup.columns:
    cols_dir = cols_dir + ['desc_tipo_acceso_local']
tabla_dir = enriq_direccion_dedup[cols_dir].copy()

# Tabla proximidad (ya no tiene duplicados)
tabla_prox = enriq_proximidad[cols_osm + cols_censo].copy()
tabla_prox['desc_tipo_acceso_local'] = np.nan

# Unir enriquecidos
enriquecidos = pd.concat([tabla_dir, tabla_prox], ignore_index=True)

# Verificar que no hay solapamiento entre dir y prox
solapamiento = set(tabla_dir['osm_id']) & set(tabla_prox['osm_id'])
print(f"Solapamiento dir/prox: {len(solapamiento)}")

# Añadir los sin enriquecer
ids_enriquecidos = set(enriquecidos['osm_id'])
sin_enriquecer = raw_osm[~raw_osm['osm_id'].isin(ids_enriquecidos)].copy()
for col in ['desc_barrio_local', 'desc_distrito_local', 'desc_epigrafe',
            'id_seccion_censal_local', 'desc_tipo_acceso_local']:
    sin_enriquecer[col] = np.nan
sin_enriquecer['metodo_match'] = 'no_matcheado'
sin_enriquecer = sin_enriquecer[enriquecidos.columns].copy()

# Universo completo
gold_base = pd.concat([enriquecidos, sin_enriquecer], ignore_index=True)

print(f"\n=== UNIVERSO COMPLETO ===")
print(f"Total: {len(gold_base)}")
print(f"osm_id únicos: {gold_base['osm_id'].nunique()}")
print(f"  Por dirección:   {(gold_base['metodo_match'] == 'direccion').sum()}")
print(f"  Por proximidad:  {(gold_base['metodo_match'] == 'proximidad').sum()}")
print(f"  Sin matchear:    {(gold_base['metodo_match'] == 'no_matcheado').sum()}")

Solapamiento dir/prox: 0

=== UNIVERSO COMPLETO ===
Total: 1688
osm_id únicos: 1688
  Por dirección:   620
  Por proximidad:  737
  Sin matchear:    331


In [11]:
# =====================================
# BLOQUE 5: VARIABLES DE COMPETENCIA
# =====================================

from scipy.spatial import cKDTree

# Usamos las coordenadas de TODO el universo (los 1.688)
coords = gold_base[['lat', 'lon']].values

# Construir árbol de búsqueda
tree = cKDTree(coords)

# Conversión aproximada: 1 grado ≈ 111.320 metros
# Para radios en metros, convertimos a grados
radio_500m = 500 / 111320
radio_1km = 1000 / 111320

# Contar vecinos dentro de cada radio (incluye el propio restaurante, por eso restamos 1)
competencia_500m = []
competencia_1km = []

for punto in coords:
    # query_ball_point devuelve los índices dentro del radio
    vecinos_500 = len(tree.query_ball_point(punto, radio_500m)) - 1  # -1 para excluirse a sí mismo
    vecinos_1km = len(tree.query_ball_point(punto, radio_1km)) - 1
    competencia_500m.append(vecinos_500)
    competencia_1km.append(vecinos_1km)

gold_base['competencia_500m'] = competencia_500m
gold_base['competencia_1km'] = competencia_1km

# Ver estadísticas
print("=== VARIABLES DE COMPETENCIA ===")
print(f"\nCompetencia en 500m:")
print(f"  Media:   {gold_base['competencia_500m'].mean():.1f}")
print(f"  Mediana: {gold_base['competencia_500m'].median():.0f}")
print(f"  Máximo:  {gold_base['competencia_500m'].max()}")
print(f"  Mínimo:  {gold_base['competencia_500m'].min()}")

print(f"\nCompetencia en 1km:")
print(f"  Media:   {gold_base['competencia_1km'].mean():.1f}")
print(f"  Mediana: {gold_base['competencia_1km'].median():.0f}")
print(f"  Máximo:  {gold_base['competencia_1km'].max()}")

# Ver los 5 restaurantes con más competencia
print(f"\n=== TOP 5 CON MÁS COMPETENCIA EN 500m ===")
top_comp = gold_base.nlargest(5, 'competencia_500m')[['name', 'desc_barrio_local', 'competencia_500m', 'competencia_1km']]
print(top_comp.to_string())

=== VARIABLES DE COMPETENCIA ===

Competencia en 500m:
  Media:   208.8
  Mediana: 235
  Máximo:  350
  Mínimo:  0

Competencia en 1km:
  Media:   654.7
  Mediana: 688
  Máximo:  1092

=== TOP 5 CON MÁS COMPETENCIA EN 500m ===
                         name     desc_barrio_local  competencia_500m  competencia_1km
983              Cuzco Lupita  SOL                                350              946
128      La Gloria de Montera  SOL                                349              973
293     El Rincón de Pangpang  SOL                                348              990
1137     El Castizo de Alcalá  SOL                                348              923
138   La Venganza de Malinche  SOL                                346              993


In [12]:
# =====================================
# CONSTRUIR LA TABLA GOLD FINAL
# =====================================

import hashlib
from datetime import datetime

def generar_id(row):
    """Genera un ID sintético reproducible: hash de nombre + CP + coordenadas."""
    base = f"{row['name']}_{row['addr_postcode']}_{round(row['lat'],5)}_{round(row['lon'],5)}"
    return hashlib.md5(base.encode()).hexdigest()[:12]

# Construir la tabla gold con los nombres finales del diseño
gold = pd.DataFrame()

# --- Bloque 1: Identificación y localización ---
gold['restaurante_id'] = gold_base.apply(generar_id, axis=1)
gold['nombre'] = gold_base['name']
gold['latitud'] = gold_base['lat']
gold['longitud'] = gold_base['lon']
gold['direccion'] = gold_base['addr_street']
gold['numero'] = gold_base['addr_housenumber']
gold['codigo_postal'] = gold_base['addr_postcode']
gold['barrio'] = gold_base['desc_barrio_local']
gold['distrito'] = gold_base['desc_distrito_local']
gold['seccion_censal'] = gold_base['id_seccion_censal_local']

# --- Bloque 2: Clasificación ---
gold['epigrafe_oficial'] = gold_base['desc_epigrafe']
gold['cocina'] = gold_base['cuisine']
gold['tipo_acceso'] = gold_base['desc_tipo_acceso_local']

# --- Bloque 3: Características operativas ---
gold['tiene_web'] = gold_base['website'].notna()
gold['url_web'] = gold_base['website']
gold['tiene_telefono'] = gold_base['phone'].notna()
gold['tiene_horario'] = gold_base['opening_hours'].notna()
gold['tiene_terraza'] = gold_base['outdoor_seating'].notna()
gold['ofrece_delivery'] = gold_base['delivery'].notna()
gold['ofrece_takeaway'] = gold_base['takeaway'].notna()
gold['es_accesible'] = gold_base['wheelchair'].notna()

# --- Bloque 5: Geográfico derivado ---
gold['competencia_500m'] = gold_base['competencia_500m']
gold['competencia_1km'] = gold_base['competencia_1km']

# --- Bloque 7: Metadatos ---
gold['fuente_principal'] = 'OSM'
gold['metodo_enriquecimiento_censo'] = gold_base['metodo_match']
gold['fecha_construccion'] = datetime.now().strftime('%Y-%m-%d')

print(f"✅ Tabla gold construida: {len(gold)} filas × {len(gold.columns)} columnas")
print(f"\nColumnas: {list(gold.columns)}")
print(f"\n=== PRIMERAS 3 FILAS ===")
gold.head(3)

✅ Tabla gold construida: 1688 filas × 26 columnas

Columnas: ['restaurante_id', 'nombre', 'latitud', 'longitud', 'direccion', 'numero', 'codigo_postal', 'barrio', 'distrito', 'seccion_censal', 'epigrafe_oficial', 'cocina', 'tipo_acceso', 'tiene_web', 'url_web', 'tiene_telefono', 'tiene_horario', 'tiene_terraza', 'ofrece_delivery', 'ofrece_takeaway', 'es_accesible', 'competencia_500m', 'competencia_1km', 'fuente_principal', 'metodo_enriquecimiento_censo', 'fecha_construccion']

=== PRIMERAS 3 FILAS ===


,restaurante_id,nombre,latitud,longitud,direccion,numero,codigo_postal,barrio,distrito,seccion_censal,...,tiene_horario,tiene_terraza,ofrece_delivery,ofrece_takeaway,es_accesible,competencia_500m,competencia_1km,fuente_principal,metodo_enriquecimiento_censo,fecha_construccion
0,b8c3a2fd9249,La Casa del Abuelo,40.414574,-3.707499,Calle de Toledo,11,28005,SOL,CENTRO,1121.0,...,False,True,False,False,True,302,838,OSM,direccion,2026-09-17
1,aa5582e14325,Barinka,40.414802,-3.704108,Calle de la Bolsa,4,None,SOL,CENTRO,1120.0,...,True,False,False,False,False,269,993,OSM,direccion,2026-09-17
2,b4f5159db94f,Taberna Griega,40.425155,-3.704470,Calle del Tesoro,6,28004,UNIVERSIDAD,CENTRO,1100.0,...,True,False,False,False,False,214,727,OSM,direccion,2026-09-17


In [13]:
# =====================================
# CHEQUEO DE CALIDAD DE LA CAPA GOLD
# =====================================

print("=== COMPLETITUD DE CAMPOS (% no nulos) ===\n")
completitud = (gold.notna().sum() / len(gold) * 100).round(1)
for col, pct in completitud.items():
    print(f"  {col:35s} {pct:5.1f}%")

print("\n=== ENRIQUECIMIENTO DESDE CENSO ===")
print(gold['metodo_enriquecimiento_censo'].value_counts())

print("\n=== DISTRIBUCIÓN POR BARRIO (top 8) ===")
print(gold['barrio'].value_counts().head(8))

print("\n=== TIPOS DE COCINA (top 10) ===")
print(gold['cocina'].value_counts().head(10))

print("\n=== VARIABLES BOOLEANAS (% True) ===")
for col in ['tiene_web', 'tiene_telefono', 'tiene_horario', 'tiene_terraza', 'ofrece_delivery', 'ofrece_takeaway', 'es_accesible']:
    pct = (gold[col].sum() / len(gold) * 100)
    print(f"  {col:20s} {pct:5.1f}%")

=== COMPLETITUD DE CAMPOS (% no nulos) ===

  restaurante_id                      100.0%
  nombre                               98.6%
  latitud                             100.0%
  longitud                            100.0%
  direccion                            78.8%
  numero                               74.6%
  codigo_postal                        69.1%
  barrio                               80.4%
  distrito                             80.4%
  seccion_censal                       80.4%
  epigrafe_oficial                     80.4%
  cocina                               54.1%
  tipo_acceso                          36.7%
  tiene_web                           100.0%
  url_web                              41.9%
  tiene_telefono                      100.0%
  tiene_horario                       100.0%
  tiene_terraza                       100.0%
  ofrece_delivery                     100.0%
  ofrece_takeaway                     100.0%
  es_accesible                        100.0%
  competenc

In [14]:
# =====================================
# GUARDAR LA CAPA GOLD
# =====================================

ruta_gold_parquet = GOLD_DIR / "gold_restaurantes_madrid.parquet"
ruta_gold_csv = GOLD_DIR / "gold_restaurantes_madrid.csv"

gold.to_parquet(ruta_gold_parquet, index=False)
gold.to_csv(ruta_gold_csv, index=False)

print(f"✅ Capa gold guardada:")
print(f"   {ruta_gold_parquet}")
print(f"   {ruta_gold_csv}")
print(f"\n   {len(gold)} restaurantes × {len(gold.columns)} columnas")
print(f"   Tamaño Parquet: {ruta_gold_parquet.stat().st_size / 1024:.1f} KB")
print(f"   Tamaño CSV: {ruta_gold_csv.stat().st_size / 1024:.1f} KB")

✅ Capa gold guardada:
   ../data/gold/gold_restaurantes_madrid.parquet
   ../data/gold/gold_restaurantes_madrid.csv

   1688 restaurantes × 26 columnas
   Tamaño Parquet: 126.3 KB
   Tamaño CSV: 367.1 KB


In [15]:
# =====================================
# INSPECCIONAR CÓDIGOS DE SECCIÓN CENSAL
# =====================================

# Ver cómo está la sección censal en nuestra capa gold
print("=== SECCIÓN CENSAL EN NUESTRA CAPA GOLD ===")
print(gold[gold['seccion_censal'].notna()]['seccion_censal'].head(15))
print(f"\nTipo de dato: {gold['seccion_censal'].dtype}")
print(f"Valores únicos (muestra): {sorted(gold['seccion_censal'].dropna().unique())[:20]}")

=== SECCIÓN CENSAL EN NUESTRA CAPA GOLD ===
0     1121.0
1     1120.0
2     1100.0
3     1116.0
4     1121.0
5     1114.0
6     1115.0
7     1069.0
8     1081.0
9     1076.0
10    1011.0
11    1103.0
12    1105.0
13    1096.0
14    1027.0
Name: seccion_censal, dtype: float64

Tipo de dato: float64
Valores únicos (muestra): [np.float64(1001.0), np.float64(1002.0), np.float64(1003.0), np.float64(1004.0), np.float64(1006.0), np.float64(1007.0), np.float64(1008.0), np.float64(1009.0), np.float64(1011.0), np.float64(1012.0), np.float64(1013.0), np.float64(1014.0), np.float64(1015.0), np.float64(1016.0), np.float64(1018.0), np.float64(1019.0), np.float64(1020.0), np.float64(1021.0), np.float64(1022.0), np.float64(1023.0)]


In [17]:
# =====================================
# CARGAR INE Y VERIFICAR MAPEO DE SECCIÓN CENSAL
# =====================================

# Cargar el fichero del INE (encoding latin-1, separador ;)
ine = pd.read_csv(
    "../data/raw/ine_renta_seccion_madrid.csv",
    sep=';',
    encoding='latin-1'
)

print(f"INE cargado: {len(ine)} filas")
print(f"Columnas: {list(ine.columns)}")
print(f"\n=== Primeras filas de Madrid distrito 01 (Centro) ===")
madrid_centro = ine[ine['Secciones'].str.startswith('2807901', na=False)]
print(madrid_centro[['Secciones', 'Total']].head(10).to_string())

print(f"\nSecciones del distrito Centro en INE: {len(madrid_centro)}")

INE cargado: 4541 filas
Columnas: ['Municipios', 'Distritos', 'Secciones', 'Indicadores de renta media y mediana', 'Periodo', 'Total']

=== Primeras filas de Madrid distrito 01 (Centro) ===
                            Secciones   Total
1108  2807901001 Madrid sección 01001  27.408
1109  2807901002 Madrid sección 01002  20.951
1110  2807901003 Madrid sección 01003  21.921
1111  2807901004 Madrid sección 01004  32.264
1112  2807901006 Madrid sección 01006  28.844
1113  2807901007 Madrid sección 01007  25.414
1114  2807901008 Madrid sección 01008  29.172
1115  2807901009 Madrid sección 01009  24.639
1116  2807901011 Madrid sección 01011  22.651
1117  2807901012 Madrid sección 01012  19.722

Secciones del distrito Centro en INE: 111
